# Проверка скелетов ветвей (данные для модулей T и P)

Ноутбук проверяет, пригодны ли исходные `branch_skeleton.npy` для построения центральной линии ствола и профиля радиуса (`docs/neuron-model/module-interfaces-spec.md`, разделы 3.3 и 4; `docs/neuron-model/tp-modules-literature-review.md`, раздел 4.1).

Для нескольких ветвей с наибольшим числом шипиков:
1. диагностика скелета: связность, циклы, точки ветвления, доля наидлиннейшего пути в общей длине, длины отрезков;
2. согласованность скелета и меша: расстояние от вершин меша до скелета, расстояние от скелета до поверхности;
3. 3D-график: меш ветви (полупрозрачный), все отрезки скелета, наидлиннейший путь (кандидат в центральную линию), точки ветвления и концы;
4. пробный профиль радиуса $r(s)$ вдоль наидлиннейшего пути — по всем вершинам меша и после исключения вершин шипиков.

Данные — локальная тестовая выборка `datasets/minnie65/<neuron_id>/` (mac: только несколько ветвей, один процесс).

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() and (candidate / "AGENTS.md").exists():
            return candidate
    raise RuntimeError("Repository root was not found.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.spatial import cKDTree

from src.neuron_model.spine_geometry import (
    graph_from_segments,
    load_skeleton_segments,
    load_trimesh,
    longest_path_polyline,
    sample_polyline_by_arclength,
)
from src.neuron_model.visualization import make_scene_figure, mesh_trace

GRAY, PINK, GREEN = "#B5B8BC", "#FD89C3", "#8CBD3A"

## Параметры

In [ ]:
DATASET_ROOT = REPO_ROOT / "datasets" / "minnie65"
NEURON_ID = "864691134886335738"
N_BRANCHES = 3              # сколько ветвей с наибольшим числом шипиков показать
BRANCHES = None             # или явный список [("000", "000"), ...] = (limb, branch)
N_RADIUS_SAMPLES = 200      # точек вдоль наидлиннейшего пути для профиля радиуса
RADIUS_SLAB_NM = 150.0      # полуширина слоя, перпендикулярного касательной
RADIUS_MAX_NM = 5000.0      # вершины дальше этого от центральной линии в оценку не идут

## Выбор ветвей

In [ ]:
def branch_dir(limb, branch):
    return DATASET_ROOT / NEURON_ID / f"limb_{limb}" / f"branch_{branch}"


def list_branches_by_spine_count():
    rows = []
    for limb_dir in sorted((DATASET_ROOT / NEURON_ID).glob("limb_*")):
        for bdir in sorted(limb_dir.glob("branch_*")):
            if not (bdir / "branch_skeleton.npy").exists():
                continue
            rows.append({
                "limb": limb_dir.name.split("_")[1],
                "branch": bdir.name.split("_")[1],
                "n_spines": len(list((bdir / "spines").glob("spine_*.off"))),
            })
    return pd.DataFrame(rows).sort_values("n_spines", ascending=False).reset_index(drop=True)


branch_table = list_branches_by_spine_count()
selected = BRANCHES or list(branch_table.head(N_BRANCHES)[["limb", "branch"]].itertuples(index=False, name=None))
print(f"ветвей в нейроне: {len(branch_table)}; выбраны: {selected}")
branch_table.head(10)

## Диагностика скелета и согласованность с мешем

In [ ]:
def sample_segments(segments, step_nm=20.0):
    """Плотная выборка точек вдоль всех отрезков скелета (для KD-дерева)."""
    points = []
    for start, end in segments:
        n = max(2, int(np.ceil(np.linalg.norm(end - start) / step_nm)) + 1)
        t = np.linspace(0.0, 1.0, n)[:, None]
        points.append(start * (1 - t) + end * t)
    return np.vstack(points)


def polyline_length(polyline):
    return float(np.linalg.norm(np.diff(polyline, axis=0), axis=1).sum())


def skeleton_diagnostics(segments, mesh):
    graph = graph_from_segments(segments)
    degrees = np.array([d for _, d in graph.degree()])
    seg_lengths = np.array([np.linalg.norm(e - s) for s, e in segments])
    total_length = float(seg_lengths.sum())
    centerline = longest_path_polyline(segments)
    longest = polyline_length(centerline)

    skel_points = sample_segments(segments)
    tree = cKDTree(skel_points)
    vertex_to_skel, _ = tree.query(np.asarray(mesh.vertices))
    surf_tree = cKDTree(np.asarray(mesh.vertices))
    skel_to_surface, _ = surf_tree.query(skel_points)

    stats = {
        "n_segments": len(segments),
        "n_nodes": graph.number_of_nodes(),
        "n_components": nx.number_connected_components(graph),
        "n_cycles": len(nx.cycle_basis(graph)),
        "n_terminals(deg=1)": int((degrees == 1).sum()),
        "n_branch_points(deg>=3)": int((degrees >= 3).sum()),
        "total_length_um": total_length / 1000,
        "longest_path_um": longest / 1000,
        "longest/total": longest / total_length if total_length else np.nan,
        "seg_len_median_nm": float(np.median(seg_lengths)),
        "seg_len_max_nm": float(seg_lengths.max()),
        "mesh_vertices": len(mesh.vertices),
        "vertex→skel median_nm": float(np.median(vertex_to_skel)),
        "vertex→skel p99_nm": float(np.percentile(vertex_to_skel, 99)),
        "skel→surface median_nm": float(np.median(skel_to_surface)),
        "skel→surface max_nm": float(skel_to_surface.max()),
    }
    return stats, graph, centerline


results = {}
for limb, branch in selected:
    bdir = branch_dir(limb, branch)
    segments = load_skeleton_segments(bdir / "branch_skeleton.npy")
    mesh = load_trimesh(bdir / "branch_mesh.off", process=False)
    stats, graph, centerline = skeleton_diagnostics(segments, mesh)
    results[(limb, branch)] = dict(segments=segments, mesh=mesh, graph=graph, centerline=centerline, stats=stats)

pd.DataFrame({f"{l}/{b}": r["stats"] for (l, b), r in results.items()})

Как читать таблицу:
- `n_components = 1`, `n_cycles = 0` — скелет связный и без циклов, наидлиннейший путь определён однозначно;
- `longest/total` близко к 1 — скелет почти неветвящийся; заметно меньше 1 — есть боковые ответвления (шипики/артефакты), которые отсекает выбор наидлиннейшего пути;
- `skel→surface median` — грубая оценка радиуса ствола (скелет должен лежать внутри меша, на расстоянии порядка радиуса от поверхности);
- `vertex→skel p99` — насколько далеко от скелета уходят вершины меша (верхний хвост — головки шипиков).

## 3D: меш ветви и скелет

In [ ]:
def segment_lines_trace(segments, *, name, color, width):
    xs, ys, zs = [], [], []
    for s, e in segments:
        xs += [s[0], e[0], None]
        ys += [s[1], e[1], None]
        zs += [s[2], e[2], None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode="lines", name=name, line=dict(color=color, width=width))


def node_markers(graph, degree_filter, *, name, color, size):
    pts = np.array([graph.nodes[n]["pos"] for n, d in graph.degree() if degree_filter(d)])
    if len(pts) == 0:
        return None
    return go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="markers", name=name,
                        marker=dict(size=size, color=color))


for (limb, branch), r in results.items():
    centerline = r["centerline"]
    traces = [
        mesh_trace(r["mesh"], name="branch mesh", color=GRAY, opacity=0.25),
        segment_lines_trace(r["segments"], name="skeleton (все отрезки)", color="black", width=3),
        go.Scatter3d(x=centerline[:, 0], y=centerline[:, 1], z=centerline[:, 2], mode="lines",
                     name="наидлиннейший путь", line=dict(color=PINK, width=7)),
    ]
    for trace in (
        node_markers(r["graph"], lambda d: d >= 3, name="точки ветвления", color="red", size=4),
        node_markers(r["graph"], lambda d: d == 1, name="концы", color=GREEN, size=4),
    ):
        if trace is not None:
            traces.append(trace)
    fig = make_scene_figure(traces, title=f"{NEURON_ID} limb_{limb}/branch_{branch}", width=900, height=700)
    fig.update_layout(showlegend=True)
    fig.show()

## Пробный профиль радиуса $r(s)$

Радиус в точке центральной линии — медиана расстояний от вершин меша до центральной линии в тонком слое, перпендикулярном касательной. Меш ветви содержит шипики, поэтому оценка считается дважды: по всем вершинам и после исключения вершин шипиков. Вершины шипиков находятся по совпадению координат с вершинами мешей из `spines/` (меши шипиков вырезаны из меша ветви).

In [ ]:
def spine_vertex_mask(mesh, bdir, decimals=3):
    keys = set()
    for spine_path in (bdir / "spines").glob("spine_*.off"):
        spine = load_trimesh(spine_path, process=False)
        keys.update(map(tuple, np.round(np.asarray(spine.vertices), decimals)))
    verts = np.round(np.asarray(mesh.vertices), decimals)
    return np.fromiter((tuple(v) in keys for v in verts), dtype=bool, count=len(verts))


def radius_profile(centerline, vertices, n_samples, slab, max_r):
    samples = sample_polyline_by_arclength(centerline, n_samples)
    tangents = np.gradient(samples, axis=0)
    tangents /= np.linalg.norm(tangents, axis=1, keepdims=True)
    s = np.r_[0.0, np.cumsum(np.linalg.norm(np.diff(samples, axis=0), axis=1))]
    radius = np.full(len(samples), np.nan)
    for j, (p, t) in enumerate(zip(samples, tangents)):
        d = vertices - p
        along = d @ t
        perp = np.linalg.norm(d - along[:, None] * t, axis=1)
        sel = (np.abs(along) < slab) & (perp < max_r)
        if sel.sum() >= 5:
            radius[j] = np.median(perp[sel])
    return s, radius


for (limb, branch), r in results.items():
    bdir = branch_dir(limb, branch)
    vertices = np.asarray(r["mesh"].vertices)
    mask = spine_vertex_mask(r["mesh"], bdir)
    s, r_all = radius_profile(r["centerline"], vertices, N_RADIUS_SAMPLES, RADIUS_SLAB_NM, RADIUS_MAX_NM)
    _, r_shaft = radius_profile(r["centerline"], vertices[~mask], N_RADIUS_SAMPLES, RADIUS_SLAB_NM, RADIUS_MAX_NM)
    print(f"limb_{limb}/branch_{branch}: вершин шипиков {mask.sum()} из {len(mask)} ({mask.mean():.1%}); "
          f"медианный радиус: все вершины {np.nanmedian(r_all):.0f} нм, только ствол {np.nanmedian(r_shaft):.0f} нм")

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=s / 1000, y=r_all, mode="lines", name="все вершины", line=dict(color=GRAY)))
    fig.add_trace(go.Scatter(x=s / 1000, y=r_shaft, mode="lines", name="без шипиков", line=dict(color=PINK, width=3)))
    fig.update_layout(title=f"r(s): limb_{limb}/branch_{branch}", xaxis_title="s, мкм (вдоль наидлиннейшего пути)",
                      yaxis_title="радиус, нм", width=900, height=400)
    fig.show()

## Лежит ли скелет на оси ствола?

Для центральной линии ствола скелет должен проходить внутри меша, примерно посередине сечения. Проверка по точкам наидлиннейшего пути:
- обобщённое число оборотов (winding number) относительно меша: $>0.5$ — точка внутри;
- расстояние до поверхности (точка — треугольник);
- смещение точки скелета от центра поперечного сечения ствола.

Центр и радиус сечения оцениваются так:
1. **Касательная** берётся по точкам скелета на $\pm$`TANGENT_HALF_WINDOW` отсчётов (около $\pm 2$ мкм), а не по соседним точкам. Исходный скелет идёт по поверхности и локально «дрожит»; по соседним точкам сечение получается косым и вместо кольца даёт полосу.
2. **В срез** попадают вершины ствола (без вершин шипиков) в слое толщиной $2\times$`RADIUS_SLAB_NM`, не дальше `CENTER_MAX_R_NM` от точки скелета.
3. **Из среза берётся только связный кусок**, ближайший к точке скелета: точки, соединённые цепочкой соседей ближе `CLUSTER_EPS_NM`. Скелет лежит на поверхности ствола, поэтому этот кусок — кольцо ствола. Отдельные посторонние фрагменты (соседний изгиб ветви, отдельно стоящие выросты) отбрасываются.
4. **Окружность** подгоняется методом Kåsa (наименьшие квадраты) с итеративным отбрасыванием точек, отстоящих от окружности дальше `ROBUST_K`$\times r$. Это подавляет вклад неотсегментированных шеек и выростов, примыкающих к кольцу. Подгонка окружности, в отличие от центроида, устойчива к неполному кольцу вершин (вырезанные основания шипиков, разрезы меша на концах ветви).

In [ ]:
import scipy.sparse as sps
import trimesh
from scipy.sparse.csgraph import connected_components
from src.neuron_model.spine_sampling import winding_numbers

TANGENT_HALF_WINDOW = 3
CENTER_MAX_R_NM = 1200.0
CLUSTER_EPS_NM = 150.0
ROBUST_K = 0.35


def fit_circle_2d(xy):
    """Алгебраическая подгонка окружности (Kåsa): x^2 + y^2 = 2ax + 2by + c."""
    A = np.c_[2 * xy[:, 0], 2 * xy[:, 1], np.ones(len(xy))]
    rhs = (xy ** 2).sum(axis=1)
    (a, b, c), *_ = np.linalg.lstsq(A, rhs, rcond=None)
    return np.array([a, b]), float(np.sqrt(max(c + a * a + b * b, 0.0)))


def robust_fit_circle_2d(xy, k=ROBUST_K, iterations=5):
    keep = np.ones(len(xy), dtype=bool)
    for _ in range(iterations):
        center, radius = fit_circle_2d(xy[keep])
        new = np.abs(np.linalg.norm(xy - center, axis=1) - radius) < k * radius
        if new.sum() < 10 or (new == keep).all():
            break
        keep = new
    return center, radius


def nearest_cluster(xy, eps=CLUSTER_EPS_NM):
    """Маска связного куска точек среза, ближайшего к точке скелета (начало координат среза)."""
    pairs = cKDTree(xy).query_pairs(eps, output_type="ndarray")
    graph = sps.coo_matrix((np.ones(len(pairs)), (pairs[:, 0], pairs[:, 1])), shape=(len(xy), len(xy)))
    _, labels = connected_components(graph, directed=False)
    return labels == labels[np.argmin(np.linalg.norm(xy, axis=1))]


def smoothed_tangents(points, half_window=TANGENT_HALF_WINDOW):
    n = len(points)
    t = np.array([points[min(j + half_window, n - 1)] - points[max(j - half_window, 0)] for j in range(n)])
    return t / np.linalg.norm(t, axis=1, keepdims=True)


def section_frame(t):
    e1 = np.cross(t, [1.0, 0.0, 0.0])
    if np.linalg.norm(e1) < 0.1:
        e1 = np.cross(t, [0.0, 1.0, 0.0])
    e1 /= np.linalg.norm(e1)
    return e1, np.cross(t, e1)


def recentered_centerline(centerline, shaft_vertices, n_samples, slab, max_r=CENTER_MAX_R_NM):
    """Центры и радиусы поперечных сечений ствола вдоль центральной линии."""
    samples = sample_polyline_by_arclength(centerline, n_samples)
    tangents = smoothed_tangents(samples)
    centers = np.full_like(samples, np.nan)
    radii = np.full(len(samples), np.nan)
    for j, (p, t) in enumerate(zip(samples, tangents)):
        e1, e2 = section_frame(t)
        d = shaft_vertices - p
        xy = np.c_[d @ e1, d @ e2]
        xy = xy[(np.abs(d @ t) < slab) & (np.linalg.norm(xy, axis=1) < max_r)]
        if len(xy) < 10:
            continue
        xy = xy[nearest_cluster(xy)]
        if len(xy) < 10:
            continue
        center2d, r = robust_fit_circle_2d(xy)
        centers[j] = p + center2d[0] * e1 + center2d[1] * e2
        radii[j] = r
    return samples, tangents, centers, radii


for (limb, branch), r in results.items():
    bdir = branch_dir(limb, branch)
    mesh = r["mesh"]
    vertices = np.asarray(mesh.vertices)
    mask = spine_vertex_mask(mesh, bdir)
    samples, tangents, centers, radii = recentered_centerline(r["centerline"], vertices[~mask], N_RADIUS_SAMPLES,
                                                               RADIUS_SLAB_NM)
    w = winding_numbers(samples, np.asarray(mesh.triangles))
    _, dist_surface, _ = trimesh.proximity.closest_point(mesh, samples)
    ok = ~np.isnan(radii)
    offset = np.linalg.norm(samples[ok] - centers[ok], axis=1)
    w_centers = winding_numbers(centers[ok], np.asarray(mesh.triangles))
    w_all = np.full(len(centers), np.nan)
    w_all[ok] = w_centers
    r.update(samples=samples, tangents=tangents, centers=centers, radii=radii, mask=mask, w_centers=w_all)
    print(f"limb_{limb}/branch_{branch}:")
    print(f"  исходный скелет: внутри меша {np.mean(w > 0.5):.0%} точек; "
          f"до поверхности медиана {np.median(dist_surface):.0f} нм")
    print(f"  смещение скелета от центра сечения: медиана {np.median(offset):.0f} нм, "
          f"медиана offset/radius {np.median(offset / radii[ok]):.2f}")
    print(f"  центры сечений: внутри меша {np.mean(w_centers > 0.5):.0%}; "
          f"радиус по окружности: медиана {np.nanmedian(radii):.0f} нм")

На 3D-графике грани меша, все вершины которых принадлежат мешам шипиков из `spines/`, выделены зелёным. Серые выступы на стволе — кандидаты в неотсегментированные шипики: их вершины считаются вершинами ствола и тянут на себя подгонку окружности.

Отмеченные сечения (наведите курсор, чтобы увидеть номер сечения $j$, радиус, излом и winding number):
- **красные** — излом: центр сечения отклоняется от середины между соседними центрами дальше `KINK_FACTOR` × медианный радиус, или радиус больше `SUSPECT_RADIUS_FACTOR` × медианный радиус;
- **оранжевые** — центр сечения оказался вне меша (winding number $<0.5$).

Излом считается по соседним центрам, а не по скользящему среднему: на изогнутой ветви скользящее среднее по нескольким микрометрам само отходит от кривой и ложно помечает естественные изгибы.

In [ ]:
from src.neuron_model.visualization import face_color_map

SUSPECT_RADIUS_FACTOR = 1.5
KINK_FACTOR = 0.5

for (limb, branch), r in results.items():
    ok = ~np.isnan(r["radii"])
    idx = np.flatnonzero(ok)
    centers = r["centers"][ok]
    radii = r["radii"][ok]
    w = r["w_centers"][ok]
    cl = r["centerline"]

    spine_faces = np.flatnonzero(r["mask"][np.asarray(r["mesh"].faces)].all(axis=1))
    median_r = np.median(radii)
    kink = np.zeros(len(centers))
    kink[1:-1] = np.linalg.norm(centers[1:-1] - 0.5 * (centers[:-2] + centers[2:]), axis=1)
    bad_kink = (kink > KINK_FACTOR * median_r) | (radii > SUSPECT_RADIUS_FACTOR * median_r)
    outside = w < 0.5
    hover = [f"j={j}, r={rr:.0f} нм, излом={k:.0f} нм, winding={ww:.2f}" for j, rr, k, ww in zip(idx, radii, kink, w)]
    print(f"limb_{limb}/branch_{branch}: граней шипиков {len(spine_faces)} из {len(r['mesh'].faces)}; "
          f"сечений с изломом/большим радиусом {bad_kink.sum()}, центров вне меша {outside.sum()} из {len(radii)}")

    traces = [
        mesh_trace(r["mesh"], name="branch mesh (зелёный — грани шипиков)", color=GRAY, opacity=0.35,
                   face_colors=face_color_map(spine_faces, GREEN)),
        go.Scatter3d(x=cl[:, 0], y=cl[:, 1], z=cl[:, 2], mode="lines", name="исходный скелет (наидлиннейший путь)",
                     line=dict(color="black", width=5)),
        go.Scatter3d(x=centers[:, 0], y=centers[:, 1], z=centers[:, 2], mode="lines+markers",
                     name="центры сечений (пересчитанная центральная линия)",
                     line=dict(color=PINK, width=6), marker=dict(size=2, color=PINK),
                     text=hover, hoverinfo="text"),
        go.Scatter3d(x=centers[bad_kink, 0], y=centers[bad_kink, 1], z=centers[bad_kink, 2], mode="markers",
                     name="излом / большой радиус", marker=dict(size=5, color="red"),
                     text=[h for h, b in zip(hover, bad_kink) if b], hoverinfo="text"),
        go.Scatter3d(x=centers[outside, 0], y=centers[outside, 1], z=centers[outside, 2], mode="markers",
                     name="центр вне меша", marker=dict(size=6, color="orange", symbol="diamond"),
                     text=[h for h, b in zip(hover, outside) if b], hoverinfo="text"),
    ]
    fig = make_scene_figure(traces, title=f"Центральная линия: limb_{limb}/branch_{branch}", width=900, height=700)
    fig.update_layout(showlegend=True)
    fig.show()

    s = np.r_[0.0, np.cumsum(np.linalg.norm(np.diff(r["samples"], axis=0), axis=1))]
    fig = go.Figure(go.Scatter(x=s / 1000, y=r["radii"], mode="lines", name="радиус сечения (окружность)",
                               line=dict(color=PINK, width=3)))
    fig.update_layout(title=f"r(s) по подгонке окружности: limb_{limb}/branch_{branch}",
                      xaxis_title="s, мкм", yaxis_title="радиус, нм", width=900, height=400)
    fig.show()

### Поперечное сечение крупным планом

2D-вид отдельного сечения в плоскости, перпендикулярной касательной: вершины ствола, вошедшие в подгонку (серые), прочие вершины ствола в срезе (светло-серые), вершины шипиков (зелёные), подогнанная окружность и её центр (розовые), точка исходного скелета (чёрная). По умолчанию для каждой ветви показывается сечение с наибольшим изломом; конкретные сечения можно задать в `INSPECT` как `(limb, branch, j)`, где `j` — номер из подсказки на 3D-графике.

In [ ]:
INSPECT = None   # например [("000", "000", 42)]


def section_points(r, j, slab=RADIUS_SLAB_NM, max_r=CENTER_MAX_R_NM):
    """Вершины среза j в плоскости сечения; used — вершины ствола из связного куска, вошедшие в подгонку."""
    p, t = r["samples"][j], r["tangents"][j]
    e1, e2 = section_frame(t)
    d = np.asarray(r["mesh"].vertices) - p
    xy = np.c_[d @ e1, d @ e2]
    sel = (np.abs(d @ t) < slab) & (np.linalg.norm(xy, axis=1) < max_r)
    xy, is_spine = xy[sel], r["mask"][sel]
    used = np.zeros(len(xy), dtype=bool)
    shaft_idx = np.flatnonzero(~is_spine)
    if len(shaft_idx) >= 10:
        used[shaft_idx[nearest_cluster(xy[shaft_idx])]] = True
    return xy, is_spine, used, (r["centers"][j] - p) @ np.c_[e1, e2]


def worst_section(r):
    ok = np.flatnonzero(~np.isnan(r["radii"]))
    c = r["centers"][ok]
    kink = np.zeros(len(ok))
    kink[1:-1] = np.linalg.norm(c[1:-1] - 0.5 * (c[:-2] + c[2:]), axis=1)
    return int(ok[np.argmax(kink)])


targets = INSPECT or [(l, b, worst_section(r)) for (l, b), r in results.items()]
for limb, branch, j in targets:
    r = results[(limb, branch)]
    xy, is_spine, used, center = section_points(r, j)
    radius = r["radii"][j]
    phi = np.linspace(0, 2 * np.pi, 200)
    other = ~is_spine & ~used
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=xy[used, 0], y=xy[used, 1], mode="markers", name="вершины ствола (в подгонке)",
                             marker=dict(color=GRAY, size=6)))
    fig.add_trace(go.Scatter(x=xy[other, 0], y=xy[other, 1], mode="markers",
                             name="вершины ствола (другой кусок среза)", marker=dict(color="lightgray", size=6)))
    fig.add_trace(go.Scatter(x=xy[is_spine, 0], y=xy[is_spine, 1], mode="markers", name="вершины шипиков",
                             marker=dict(color=GREEN, size=6)))
    fig.add_trace(go.Scatter(x=center[0] + radius * np.cos(phi), y=center[1] + radius * np.sin(phi), mode="lines",
                             name=f"окружность r={radius:.0f} нм", line=dict(color=PINK, width=3)))
    fig.add_trace(go.Scatter(x=[center[0]], y=[center[1]], mode="markers", name="центр сечения",
                             marker=dict(color=PINK, size=12, symbol="x")))
    fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers", name="точка исходного скелета",
                             marker=dict(color="black", size=10)))
    fig.update_layout(title=f"Сечение j={j}: limb_{limb}/branch_{branch}", xaxis_title="нм", yaxis_title="нм",
                      width=650, height=600, yaxis=dict(scaleanchor="x", scaleratio=1))
    fig.show()

## Что проверяем по результатам

- скелет связный, без циклов; наидлиннейший путь проходит вдоль ствола, а не уходит в шипик;
- скелет лежит внутри меша, на оси ствола. Если нет (скелет идёт вдоль поверхности), центральную линию для модуля T нужно пересчитывать по центрам поперечных сечений, а радиус — подгонкой окружности, а не как расстояние до исходного скелета;
- после исключения вершин шипиков профиль радиуса заметно ровнее и ниже, чем по всем вершинам (шипики завышают оценку);
- направление «проксимально → дистально» по одному скелету ветви не определяется — для него нужен скелет лимба/сомы (отдельная проверка).